# 3D Gaussian Splatting — Kaggle Training & Novel View Rendering

**Before running:**
- Settings (right panel) → Accelerator → **GPU T4 x2**
- Add your dataset: right panel → Add data → `nvs-gsplat-input`
- Run all cells in order

**Expected runtime:** ~30–45 min on T4 for 7k steps

## 0. Configure paths

In [ ]:
from pathlib import Path

# ── Kaggle dataset is mounted read-only at /kaggle/input/ ──
DATASET_NAME = 'nvs-gsplat-input'          # must match the dataset slug you uploaded
GSPLAT_INPUT = Path(f'/kaggle/input/{DATASET_NAME}/gsplat_input')

# ── Working dirs (writable, persisted to output) ──
LOCAL_DATA   = Path('/kaggle/working/gsplat_input')
LOCAL_RESULT = Path('/kaggle/working/outputs')
EXAMPLES_DIR = Path('/kaggle/working/gsplat_examples')

MAX_STEPS  = 7000
SH_DEGREE  = 3
TEST_EVERY = 8

LOCAL_RESULT.mkdir(parents=True, exist_ok=True)

# Verify input
assert GSPLAT_INPUT.exists(), (
    f'Dataset not found at {GSPLAT_INPUT}\n'
    f'Make sure you added the dataset in the right panel and the slug matches DATASET_NAME.'
)
sparse = GSPLAT_INPUT / 'sparse' / '0'
if not sparse.exists():
    sparse = GSPLAT_INPUT / 'sparse'
assert (sparse / 'cameras.bin').exists(), f'No cameras.bin in {sparse}'

n_images = len(list((GSPLAT_INPUT / 'images').glob('*')))
print(f'Input verified: {n_images} images, sparse model at {sparse}')

## 1. Install dependencies

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.version.cuda}')
print(f'GPU:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
assert torch.cuda.is_available(), 'GPU not available — enable it in Settings > Accelerator'

In [ ]:
%%capture
!pip install gsplat
!pip install pycolmap opencv-python numpy scipy tqdm torchmetrics[image] \
             imageio imageio-ffmpeg Pillow tensorboard matplotlib viser tyro PyYAML
!pip install fused-ssim

In [ ]:
import gsplat
print(f'gsplat: {gsplat.__version__}')

## 2. Download gsplat example scripts

In [ ]:
import subprocess, sys, urllib.request

EXAMPLES_DIR.mkdir(exist_ok=True)
(EXAMPLES_DIR / 'datasets').mkdir(exist_ok=True)

import gsplat as _gs
gsplat_version = _gs.__version__
BASE_URL = f'https://raw.githubusercontent.com/nerfstudio-project/gsplat/v{gsplat_version}/examples'

files_to_fetch = {
    'simple_trainer.py'    : f'{BASE_URL}/simple_trainer.py',
    'utils.py'             : f'{BASE_URL}/utils.py',
    'datasets/__init__.py' : None,
    'datasets/colmap.py'   : f'{BASE_URL}/datasets/colmap.py',
    'datasets/normalize.py': f'{BASE_URL}/datasets/normalize.py',
    'datasets/traj.py'     : f'{BASE_URL}/datasets/traj.py',
}

for rel_path, url in files_to_fetch.items():
    dst = EXAMPLES_DIR / rel_path
    if url is None:
        dst.write_text('')
        print(f'  Created empty {rel_path}')
    else:
        try:
            urllib.request.urlretrieve(url, dst)
            print(f'  Downloaded {rel_path}')
        except Exception as e:
            print(f'  WARN: could not fetch {rel_path}: {e}')

print(f'gsplat examples at {EXAMPLES_DIR}')

## 3. Patch downloaded scripts

In [ ]:
# ── Patch 1: simple_trainer.py — make optional viewer deps non-fatal ──────────
trainer_path = EXAMPLES_DIR / 'simple_trainer.py'
src = trainer_path.read_text()

viewer_patches = [
    ('from gsplat_viewer import GsplatViewer, GsplatRenderTabState',
     'try:\n    from gsplat_viewer import GsplatViewer, GsplatRenderTabState\nexcept ImportError:\n    GsplatViewer = None; GsplatRenderTabState = None'),
    ('from nerfview import CameraState, RenderTabState, apply_float_colormap',
     'try:\n    from nerfview import CameraState, RenderTabState, apply_float_colormap\nexcept ImportError:\n    CameraState = None; RenderTabState = None; apply_float_colormap = None'),
]
for old, new in viewer_patches:
    if old in src:
        src = src.replace(old, new)
        print(f'  simple_trainer.py: patched viewer import')

if 'GsplatViewer is None' not in src and 'disable_viewer' in src:
    src = src.replace(
        'if cfg.disable_viewer:',
        'if GsplatViewer is None or cfg.disable_viewer:'
    )
    print('  simple_trainer.py: forced disable_viewer when GsplatViewer unavailable')


# ── Patch 2: fused_ssim fallback ────────────────────────────────────────────
if 'from fused_ssim import fused_ssim' in src:
    src = src.replace(
        'from fused_ssim import fused_ssim',
        'try:\n    from fused_ssim import fused_ssim\nexcept ImportError:\n    from torchmetrics.functional.image import structural_similarity_index_measure as _ssim_base\n    def fused_ssim(a, b, padding="same"): return _ssim_base(a, b)'
    )
    print('  simple_trainer.py: patched fused_ssim fallback')
trainer_path.write_text(src)

# ── Patch 3: datasets/colmap.py — SceneManager shim for pycolmap >= 3.x ──────
# pycolmap removed SceneManager in v3.x; replace the bare import with a
# try/except that falls back to a Reconstruction-based compatibility wrapper.
SCENE_MANAGER_SHIM = '''\
try:
    from pycolmap import SceneManager
except ImportError:
    import pycolmap as _pycolmap

    class _ImageWrapper:
        def __init__(self, img):
            self._img = img
            pose = img.cam_from_world()
            self._R = pose.rotation.matrix()
            self._t = pose.translation
        def R(self): return self._R
        @property
        def tvec(self): return self._t
        @property
        def camera_id(self): return self._img.camera_id
        @property
        def name(self): return self._img.name

    class _CameraWrapper:
        def __init__(self, cam):
            self._cam = cam
            p = cam.params
            model_name = str(cam.model).split(".")[-1]
            self.camera_type = model_name
            self.width = cam.width
            self.height = cam.height
            if model_name == "SIMPLE_PINHOLE":
                self.fx = self.fy = p[0]; self.cx = p[1]; self.cy = p[2]
                self.k1 = self.k2 = self.p1 = self.p2 = 0.0
            elif model_name == "PINHOLE":
                self.fx = p[0]; self.fy = p[1]; self.cx = p[2]; self.cy = p[3]
                self.k1 = self.k2 = self.p1 = self.p2 = 0.0
            elif model_name == "SIMPLE_RADIAL":
                self.fx = self.fy = p[0]; self.cx = p[1]; self.cy = p[2]
                self.k1 = p[3]; self.k2 = self.p1 = self.p2 = 0.0
            elif model_name == "RADIAL":
                self.fx = self.fy = p[0]; self.cx = p[1]; self.cy = p[2]
                self.k1 = p[3]; self.k2 = p[4]; self.p1 = self.p2 = 0.0
            elif model_name == "OPENCV":
                self.fx = p[0]; self.fy = p[1]; self.cx = p[2]; self.cy = p[3]
                self.k1 = p[4]; self.k2 = p[5]; self.p1 = p[6]; self.p2 = p[7]
            else:
                self.fx = self.fy = p[0]
                self.cx = p[1] if len(p) > 1 else 0
                self.cy = p[2] if len(p) > 2 else 0
                self.k1 = self.k2 = self.p1 = self.p2 = 0.0

    class SceneManager:
        def __init__(self, colmap_dir):
            self._recon = _pycolmap.Reconstruction(colmap_dir)
        def load_cameras(self): pass
        def load_images(self): pass
        def load_points3D(self): pass
        @property
        def images(self):
            return {iid: _ImageWrapper(img) for iid, img in self._recon.images.items()}
        @property
        def cameras(self):
            return {cid: _CameraWrapper(cam) for cid, cam in self._recon.cameras.items()}
        @property
        def points3D(self):
            import numpy as _np
            return _np.array([p.xyz for p in self._recon.points3D.values()], dtype='float32')
        @property
        def point3D_errors(self):
            import numpy as _np
            return _np.array([p.error for p in self._recon.points3D.values()], dtype='float32')
        @property
        def point3D_colors(self):
            import numpy as _np
            return _np.array([p.color[:3] for p in self._recon.points3D.values()], dtype='uint8')
        @property
        def name_to_image_id(self):
            return {img.name: iid for iid, img in self._recon.images.items()}
        @property
        def point3D_id_to_images(self):
            return {pid: [(e.image_id, e.point2D_idx) for e in pt.track.elements]
                    for pid, pt in self._recon.points3D.items()}
        @property
        def point3D_id_to_point3D_idx(self):
            return {pid: idx for idx, pid in enumerate(self._recon.points3D.keys())}
'''

colmap_path = EXAMPLES_DIR / 'datasets' / 'colmap.py'
colmap_src  = colmap_path.read_text()

OLD_IMPORT = 'from pycolmap import SceneManager'
if OLD_IMPORT in colmap_src:
    colmap_src = colmap_src.replace(OLD_IMPORT, SCENE_MANAGER_SHIM, 1)
    colmap_path.write_text(colmap_src)
    print('  datasets/colmap.py: injected SceneManager compatibility shim')
elif 'SceneManager shim' in colmap_src or 'except ImportError' in colmap_src:
    print('  datasets/colmap.py: shim already present, skipping')
else:
    print('  datasets/colmap.py: WARNING — could not find SceneManager import to patch')

print('All patches applied.')

## 4. Copy input data to working directory

Kaggle input is read-only; we copy to `/kaggle/working/` for faster I/O during training.

In [ ]:
import shutil, time

if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)

print(f'Copying {GSPLAT_INPUT} → {LOCAL_DATA} ...')
t0 = time.time()
shutil.copytree(GSPLAT_INPUT, LOCAL_DATA)
print(f'Done in {time.time()-t0:.1f}s')

LOCAL_RESULT.mkdir(parents=True, exist_ok=True)

# COLMAP 4.x puts the sparse model at sparse/ instead of sparse/0/
# gsplat's dataset loader expects sparse/0/ — fix it here
sparse_0    = LOCAL_DATA / 'sparse' / '0'
sparse_root = LOCAL_DATA / 'sparse'
if not sparse_0.exists() and (sparse_root / 'cameras.bin').exists():
    sparse_0.mkdir(parents=True)
    for f in list(sparse_root.glob('*.bin')) + list(sparse_root.glob('*.txt')):
        shutil.copy2(f, sparse_0 / f.name)
    print('Mirrored sparse model → sparse/0/ for gsplat compatibility')

print('Input data ready.')

## 5. Probe simple_trainer.py CLI

In [ ]:
import sys, os, subprocess
sys.path.insert(0, str(EXAMPLES_DIR))

env = os.environ.copy()
env['PYTHONPATH'] = str(EXAMPLES_DIR) + ':' + env.get('PYTHONPATH', '')

result = subprocess.run(
    [sys.executable, str(EXAMPLES_DIR / 'simple_trainer.py'), '--help'],
    env=env, capture_output=True, text=True
)
for line in (result.stdout + result.stderr).splitlines()[:80]:
    print(line)

## 6. Train 3D Gaussian Splatting model

In [ ]:
import subprocess, sys, os, re
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

def run_streaming_plot(cmd, env):
    loss_steps, losses, psnrs = [], [], []

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    proc = subprocess.Popen(
        cmd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )

    for line in proc.stdout:
        step_match = re.search(r'step=(\d+)', line)
        loss_match = re.search(r'loss=([\d.]+)', line)
        psnr_match = re.search(r'psnr=([\d.]+)', line)

        if step_match and loss_match:
            step = int(step_match.group(1))
            loss = float(loss_match.group(1))
            loss_steps.append(step)
            losses.append(loss)
            if psnr_match:
                psnrs.append(float(psnr_match.group(1)))

            if step % 100 == 0:
                clear_output(wait=True)
                ax1.cla(); ax2.cla()
                ax1.plot(loss_steps, losses, 'b-', linewidth=0.8)
                ax1.set_title('Loss'); ax1.set_xlabel('Step'); ax1.set_ylabel('Loss')
                ax1.set_yscale('log')
                if psnrs:
                    ax2.plot(loss_steps[:len(psnrs)], psnrs, 'g-', linewidth=0.8)
                    ax2.set_title('PSNR (dB)'); ax2.set_xlabel('Step')
                fig.suptitle(f'Step {step} / {MAX_STEPS}')
                plt.tight_layout()
                display(fig)
                print(f'Step {step}: loss={loss:.4f}' + (f'  psnr={psnrs[-1]:.2f}' if psnrs else ''))

    proc.wait()
    plt.close(fig)
    return proc.returncode


env = os.environ.copy()
env['PYTHONPATH'] = str(EXAMPLES_DIR) + ':' + env.get('PYTHONPATH', '')

train_cmd = [
    sys.executable, str(EXAMPLES_DIR / 'simple_trainer.py'),
    'default',
    '--data-dir',               str(LOCAL_DATA),
    '--result-dir',             str(LOCAL_RESULT),
    '--data-factor',            '1',
    '--max-steps',              str(MAX_STEPS),
    '--sh-degree',              str(SH_DEGREE),
    '--ssim-lambda',            '0.2',
    '--antialiased',
    '--test-every',             str(TEST_EVERY),
    '--eval-steps',             '3000', '5000', str(MAX_STEPS),
    '--save-steps',             '3000', str(MAX_STEPS),
    '--disable-viewer',
    '--strategy.refine-start-iter', '500',
    '--strategy.refine-stop-iter',  str(min(5000, MAX_STEPS - 500)),
    '--strategy.refine-every',      '100',
    '--strategy.reset-every',       '3000',
    '--strategy.grow-grad2d',       '0.0002',
    '--strategy.grow-scale3d',      '0.01',
    '--strategy.prune-scale3d',     '0.1',
]

rc = run_streaming_plot(train_cmd, env)
if rc != 0:
    print(f'ERROR: training exited with code {rc}')
    diag = subprocess.run(
        [sys.executable, str(EXAMPLES_DIR / 'simple_trainer.py'), 'default', '--help'],
        env=env, capture_output=True, text=True
    )
    for line in (diag.stdout + diag.stderr).splitlines()[:60]:
        print(line)
else:
    print('Training complete!')


## 7. Render novel views

Generates views along three camera trajectories:
- **ellipse**: smooth orbit around the scene
- **interpolated**: B-spline through training poses
- **spiral**: zooming spiral

In [ ]:
from pathlib import Path

# Try working outputs first, then fall back to uploaded dataset
ckpt_dir = LOCAL_RESULT / 'ckpts'
ckpts = sorted(ckpt_dir.glob('ckpt_*.pt')) if ckpt_dir.exists() else []

if not ckpts:
    ckpts = sorted(LOCAL_RESULT.glob('**/*.pt'))

# Fall back to uploaded checkpoint dataset
if not ckpts:
    dataset_ckpt = Path('/kaggle/input/nvs-3dgs-checkpoint/ckpt_6999_rank0.pt')
    if dataset_ckpt.exists():
        ckpts = [dataset_ckpt]

if not ckpts:
    raise RuntimeError('No checkpoint found. Train first or upload checkpoint as a dataset named nvs-3dgs-checkpoint.')

checkpoint = ckpts[-1]
print(f'Using checkpoint: {checkpoint} ({checkpoint.stat().st_size/1e6:.1f} MB)')


In [ ]:
import torch
import numpy as np
from PIL import Image
from math import degrees, acos

sys.path.insert(0, str(EXAMPLES_DIR))
from datasets.colmap import Parser
from datasets.traj import (
    generate_ellipse_path_z,
    generate_interpolated_path,
    generate_spiral_path,
)
from gsplat import rasterization


def load_splats(ckpt_path: Path, device: str) -> dict:
    ckpt = torch.load(ckpt_path, map_location=device)
    splats = ckpt.get('splats', ckpt)
    return {k: v.to(device) for k, v in splats.items()}


def render_frame(splats: dict, c2w: np.ndarray, K: np.ndarray,
                 width: int, height: int, sh_degree: int, device: str) -> np.ndarray:
    c2w_np  = c2w if c2w.shape == (4, 4) else np.vstack([c2w, [0, 0, 0, 1]])
    c2w_t   = torch.from_numpy(c2w_np).float().to(device)
    viewmat = torch.linalg.inv(c2w_t).unsqueeze(0)  # (1,4,4) world-to-camera
    K_t     = torch.from_numpy(K).float().to(device).unsqueeze(0)  # (1,3,3)

    means     = splats['means']
    quats     = splats['quats']
    scales    = torch.exp(splats['scales'])           # stored in log-space
    opacities = torch.sigmoid(splats['opacities'])    # stored as logits

    if 'sh0' in splats:
        sh0 = splats['sh0']
        shN = splats.get('shN', torch.zeros(sh0.shape[0], 0, 3, device=device))
        colors = torch.cat([sh0, shN], dim=1)
    else:
        colors = splats['colors']

    render_colors, _, _ = rasterization(
        means=means, quats=quats, scales=scales,
        opacities=opacities, colors=colors,
        viewmats=viewmat, Ks=K_t,
        width=width, height=height,
        sh_degree=sh_degree,
    )
    img = render_colors[0].clamp(0, 1).detach().cpu().numpy()
    return (img * 255).astype(np.uint8)


print('Render helpers defined.')

In [ ]:
from tqdm.notebook import tqdm
import numpy as np

DEVICE        = 'cuda'
N_INPUT       = 25
PERTURB_DEG   = 12

novel_dir = LOCAL_RESULT / 'novel_views'
novel_dir.mkdir(exist_ok=True)

parser = Parser(
    data_dir=str(LOCAL_DATA),
    factor=1,
    normalize=True,
    test_every=TEST_EVERY,
)

cam_id = parser.camera_ids[0]
W, H   = parser.imsize_dict[cam_id]
K      = parser.Ks_dict[cam_id].astype(np.float32)

training_c2ws = parser.camtoworlds
n_train = len(training_c2ws)
print(f'Scene: {n_train} training poses, image {W}x{H}')

splats = load_splats(checkpoint, DEVICE)
print(f'Loaded {splats["means"].shape[0]:,} Gaussians')

selected_indices = np.linspace(0, n_train - 1, N_INPUT, dtype=int)
print(f'Selected input frames: {selected_indices.tolist()}')

def make_perturbed_c2ws(c2w, angle_deg):
    a = np.radians(angle_deg)
    rotations = [
        np.array([[ np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]]),
        np.array([[np.cos(-a), 0, np.sin(-a)], [0, 1, 0], [-np.sin(-a), 0, np.cos(-a)]]),
        np.array([[1, 0, 0], [0,  np.cos(a), -np.sin(a)], [0, np.sin(a),  np.cos(a)]]),
        np.array([[1, 0, 0], [0, np.cos(-a), -np.sin(-a)], [0, np.sin(-a), np.cos(-a)]]),
    ]
    results = []
    for R_local in rotations:
        new_c2w = c2w.copy()
        new_c2w[:3, :3] = c2w[:3, :3] @ R_local
        results.append(new_c2w)
    return results

total_rendered = 0
for rank, train_idx in enumerate(tqdm(selected_indices, desc='Input frames')):
    frame_dir = novel_dir / f'frame_{rank:02d}_trainidx{train_idx}'
    frame_dir.mkdir(exist_ok=True)

    ref_c2w = training_c2ws[train_idx]
    with torch.no_grad():
        ref_img = render_frame(splats, ref_c2w, K, W, H, SH_DEGREE, DEVICE)
    Image.fromarray(ref_img).save(frame_dir / '00_reference.png')

    for view_idx, c2w in enumerate(make_perturbed_c2ws(ref_c2w, PERTURB_DEG)):
        with torch.no_grad():
            img = render_frame(splats, c2w, K, W, H, SH_DEGREE, DEVICE)
        direction = ['left', 'right', 'up', 'down'][view_idx]
        Image.fromarray(img).save(frame_dir / f'novel_{view_idx+1:02d}_{direction}.png')
        total_rendered += 1

print(f'\nDone: {len(selected_indices)} reference frames + {total_rendered} novel views')
print(f'Saved to: {novel_dir}')


## 8. Preview renders

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

frame_dirs = sorted((LOCAL_RESULT / 'novel_views').iterdir())[:5]

fig, axes = plt.subplots(len(frame_dirs), 5, figsize=(20, 4 * len(frame_dirs)))
for row, frame_dir in enumerate(frame_dirs):
    imgs = sorted(frame_dir.glob('*.png'))
    for col, img_path in enumerate(imgs[:5]):
        axes[row][col].imshow(Image.open(img_path))
        axes[row][col].set_title(img_path.stem, fontsize=7)
        axes[row][col].axis('off')

plt.suptitle('Col 0: reference training frame | Cols 1-4: novel views (left / right / up / down)')
plt.tight_layout()
plt.show()


## Done!

Results are in `/kaggle/working/outputs/` and will appear in the **Output** tab on the right.

To download:
- Right panel → **Output** → download individual files or the full folder as a zip
- Or: File → Download Output Files

Key output files:
- `outputs/ckpts/ckpt_6999_rank0.pt` — trained model checkpoint
- `outputs/novel_views/` — rendered novel view images